# 08 · Budget del contesto e del run

Due budget: finestra corrente del modello e consumo cumulativo del run. Notebook
offline, autocontenuto, solo standard library.

## Obiettivi, prerequisiti e modalità di lettura

Distinguerai budget della finestra e budget cumulativo del run. Durata: 25–35 minuti. Tutto gira offline.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## 1 · Misurare categorie del contesto

### Spiegazione del blocco · Misura del contesto

I messaggi vengono classificati e stimati. Separare contenuto ricostruibile permette di scegliere offload prima di un riassunto costoso.

In [ ]:
from dataclasses import dataclass
from decimal import Decimal
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory

@dataclass
class Message:
    kind: str
    text: str
    reconstructible: bool = False

def tokens(text: str) -> int:
    return max(1, len(text) // 4)

messages = [Message("system", "regole " * 100), Message("tool", "X" * 8_000, True)]
total = sum(tokens(item.text) for item in messages)
reconstructible = sum(tokens(item.text) for item in messages if item.reconstructible)
print({"total": total, "reconstructible": reconstructible})

### Output atteso

Dizionario con circa `total=2175` e `reconstructible=2000`.

## 2 · Decidere: keep, offload, compact, stop

### Spiegazione del blocco · Policy di riduzione

La decisione usa rapporto di riempimento e quantità ricostruibile. L'ordine rende esplicito perché si preferisce offload a compaction.

In [ ]:
def decide(total: int, usable: int, reconstructible: int) -> str:
    ratio = total / usable
    if ratio >= 1:
        return "stop"
    if ratio >= 0.75 and reconstructible > 500:
        return "offload"
    if ratio >= 0.75:
        return "compact"
    return "keep"

action = decide(total, usable=2_400, reconstructible=reconstructible)
print("Azione:", action)
assert action == "offload"

### Output atteso

`Azione: offload`; l'assert passa.

## 3 · Offload recuperabile: file + checksum + estratto

### Spiegazione del blocco · Offload con integrità

Il contenuto completo finisce su file; nel contesto resta riferimento, checksum ed estratto. Il checksum viene verificato subito.

In [ ]:
with TemporaryDirectory() as temporary:
    target = Path(temporary) / "tool-output.txt"
    full = messages[-1].text
    target.write_text(full, encoding="utf-8")
    digest = sha256(full.encode()).hexdigest()[:16]
    replacement = f"[offloaded] ref={target} sha256={digest} excerpt={full[:80]}…"
    assert sha256(target.read_bytes()).hexdigest().startswith(digest)
    print(replacement)

### Output atteso

Una riga `[offloaded]` con path temporaneo, SHA-256 abbreviato ed estratto.

## 4 · Prenotare budget prima della chiamata

### Spiegazione del blocco · Ledger preventivo

La prenotazione valuta massimo teorico prima della chiamata; `settle` registra consumo effettivo. Questo evita sforamenti dovuti a call concorrenti.

In [ ]:
@dataclass
class Ledger:
    max_tokens: int
    max_cost: Decimal
    used_tokens: int = 0
    used_cost: Decimal = Decimal(0)

    def reserve(self, input_tokens: int, output_tokens: int, rate_in: Decimal, rate_out: Decimal):
        projected_tokens = self.used_tokens + input_tokens + output_tokens
        projected_cost = self.used_cost + (Decimal(input_tokens) * rate_in + Decimal(output_tokens) * rate_out) / Decimal(1_000_000)
        if projected_tokens > self.max_tokens or projected_cost > self.max_cost:
            raise RuntimeError("budget_exceeded")
        return input_tokens, output_tokens, projected_cost - self.used_cost

    def settle(self, reservation, actual_input: int, actual_output: int, rate_in: Decimal, rate_out: Decimal):
        self.used_tokens += actual_input + actual_output
        self.used_cost += (Decimal(actual_input) * rate_in + Decimal(actual_output) * rate_out) / Decimal(1_000_000)

ledger = Ledger(10_000, Decimal("0.10"))
reservation = ledger.reserve(2_000, 1_000, Decimal("1"), Decimal("6"))
ledger.settle(reservation, 2_000, 300, Decimal("1"), Decimal("6"))
print(ledger)

### Output atteso

Rappresentazione di `Ledger` con `used_tokens=2300` e costo calcolato.

## Prova tu

Simula due prenotazioni parallele: entrambe devono considerare il budget già riservato, non solo quello consumato.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: osservare tutte le soglie

### Spiegazione del blocco

Una matrice di riempimento mostra che la policy cambia in base a pressione e ricostruibilità.

In [ ]:
for totale_demo, ricostruibile_demo in ((500, 0), (1900, 0), (1900, 900), (2500, 900)):
    print(totale_demo, ricostruibile_demo, "->", decide(totale_demo, 2400, ricostruibile_demo))

### Output atteso

Nell'ordine: `keep`, `compact`, `offload`, `stop`.

## Esempio aggiuntivo: rifiuto preventivo del budget

### Spiegazione del blocco

Il caso negativo verifica che una chiamata troppo grande venga bloccata prima di consumare risorse.

In [ ]:
ledger_piccolo = Ledger(1_000, Decimal("0.01"))
try:
    ledger_piccolo.reserve(900, 500, Decimal("1"), Decimal("6"))
except RuntimeError as errore:
    print("Bloccato prima della chiamata:", errore)

### Output atteso

`Bloccato prima della chiamata: budget_exceeded`.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.